# 07 — Initial placement vs. actual gameplay outcomes
The predictors describe only the first two placement rounds. Actual winner, final VP, and observed third-settlement timing are outcomes only. All games use RandomPlayer; conclusions are associations, not causal or human-player results.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from outcome_extraction import extract
from outcome_prediction import compare
OUT=Path("processed_data_5000")
CHARTS=Path("charts/outcomes_5000")
CHARTS.mkdir(parents=True,exist_ok=True)
o=extract(Path('data_5000'))
o.to_csv(OUT/"actual_outcomes.csv",index=False)
f=pd.read_csv(OUT/"starting_advantage.csv")
d=f.merge(o,on=["Game_ID","Player","Player_Number"],validate="one_to_one")
assert len(d)==len(f)==len(o)
assert d.loc[d.Completed_Game.eq(1)].groupby("Game_ID").Won.sum().eq(1).all()
d["Built_Third"]=d.Actual_Turns_To_Third.notna()
d.to_csv(OUT/"initial_vs_outcome.csv",index=False)
display(d.head())

In [ ]:
# Descriptive groups: fixed cutoffs inherited from the previous exploratory experiment.
# These are not fit or used as predictors for the actual-outcome model.
import numpy as np
d["Advantage_Class"]=np.select([d.Blocked,d.Turns_To_50.le(7),d.Turns_To_50.le(14)],["Blocked","Strong","Average"],default="Weak")
order=["Strong","Average","Weak","Blocked"]
summary=d.groupby("Advantage_Class").agg(Players=("Won","size"),Wins=("Won","sum"),Win_Rate=("Won","mean"),Third_Rate=("Built_Third","mean"),Mean_Final_VP=("Final_VP","mean")).reindex(order)
display(summary)
display(d.groupby("Player_Number")[["Won","Built_Third","Final_VP"]].mean())
display(d.groupby("Won")[["Total_Production","Prob_Turn_10","Turns_To_50"]].agg(["mean","median"]))
summary.to_csv(OUT/"outcome_by_advantage.csv")

In [ ]:
def chart(series,title,ylabel,name):
    ax=series.plot.bar(rot=0)
    ax.set_title(title); ax.set_ylabel(ylabel)
    ax.figure.tight_layout()
    ax.figure.savefig(CHARTS/name)
    plt.show()
chart(d.groupby("Player_Number").Won.mean()*100,"Actual wins by player order","Win rate (%)","win_by_order.svg")
chart(summary.Win_Rate*100,"Actual wins by theoretical starting advantage","Win rate (%)","win_by_advantage.svg")
chart(d.groupby("Won").Total_Production.mean().rename(index={0:"Non-winners",1:"Winners"}),"Initial production by actual outcome","Mean pips","production_vs_outcome.svg")
chart(summary.Third_Rate*100,"Third settlement by starting advantage","Built third (%)","third_by_advantage.svg")

In [ ]:
paired=d.dropna(subset=["Turns_To_50","Actual_Turns_To_Third"])
print("Paired records:",len(paired))
print("Pearson correlation:",paired.Turns_To_50.corr(paired.Actual_Turns_To_Third))
print("Observed median personal rolls among builders:",d.Actual_Turns_To_Third.median())
ax=paired.plot.scatter(x="Turns_To_50",y="Actual_Turns_To_Third",alpha=.5)
ax.set_title("Theoretical readiness vs observed third settlement")
ax.figure.tight_layout(); ax.figure.savefig(CHARTS/"readiness_vs_actual.svg"); plt.show()

The scatterplot excludes players with no observed third settlement or no finite modeled threshold; it is conditional on both outcomes. Game termination is a competing event; the missing timing is not zero. The actual third settlement can use more roads, ports and development cards, so the theoretical and observed targets differ.

In [ ]:
metrics,pred=compare(d)
display(metrics)
metrics.to_csv(OUT/"outcome_prediction_results.csv",index=False)
pred.to_csv(OUT/"outcome_predictions.csv",index=False)
ax=metrics.set_index("Model")[["ROC_AUC","Within_Game_Top1"]].plot.bar(rot=20)
ax.set_ylim(0,1); ax.set_ylabel("Score")
ax.figure.tight_layout(); ax.figure.savefig(CHARTS/"prediction_comparison.svg"); plt.show()

Five folds split by whole games using sorted game IDs modulo five. Each position receives an out-of-fold score from a regularized logistic model. Accuracy uses 0.5; ranking accuracy selects the highest score per game, sharing credit among exact ties. A uniform random winner guess has 25% ranking accuracy. No external validation or hyperparameter tuning is claimed.